#  Ejercicios de Optimización para Ingeniería de Sistemas 

In [1]:
# Instalamos la librería PuLP, que sirve para resolver problemas
# de optimización lineal y entera (solo hace falta la primera vez)
!pip install pulp -q

# Importamos la librería para poder usar sus funciones
import pulp



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\karen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


##  Ejercicio 1 — Selección óptima de proyectos de TI 

**Enunciado:**
El área de TI evalúa 4 proyectos, cada uno con un beneficio (miles de $) y un costo (miles de $):

| Proyecto | Beneficio | Costo |
|---|---|---|
| P1: Migración a la nube | 80 | 40 |
| P2: Automatización RPA | 60 | 30 |
| P3: Ciberseguridad | 90 | 50 |
| P4: App móvil interna | 45 | 25 |

Presupuesto disponible: $100,000. Cada proyecto se ejecuta completo o no se ejecuta.
¿Qué proyectos elegir para maximizar el beneficio sin exceder el presupuesto?


In [2]:
# ---------- 1. DATOS DEL PROBLEMA ----------

# Diccionario con el beneficio (en miles de $) que da cada proyecto
beneficios = {"P1": 80, "P2": 60, "P3": 90, "P4": 45}

# Diccionario con el costo (en miles de $) de cada proyecto
costos = {"P1": 40, "P2": 30, "P3": 50, "P4": 25}

# Presupuesto máximo disponible (en miles de $)
presupuesto = 100

# ---------- 2. CREAR EL MODELO ----------

# Creamos un problema de optimización llamado "Seleccion_proyectos_TI"
# pulp.LpMaximize indica que queremos MAXIMIZAR el resultado (el beneficio)
modelo = pulp.LpProblem("Seleccion_proyectos_TI", pulp.LpMaximize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# Creamos una variable binaria (0 o 1) por cada proyecto:
# valdrá 1 si decidimos ejecutar el proyecto, y 0 si no lo ejecutamos
y = {p: pulp.LpVariable(f"Ejecutar_{p}", cat="Binary") for p in beneficios}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Le decimos al modelo que la meta es maximizar la suma de los beneficios
# de los proyectos que sí se ejecuten (y[p] = 1)
modelo += pulp.lpSum(beneficios[p] * y[p] for p in beneficios), "Beneficio_total"

# ---------- 5. RESTRICCIÓN ----------

# La suma de los costos de los proyectos elegidos no puede superar el presupuesto
modelo += pulp.lpSum(costos[p] * y[p] for p in beneficios) <= presupuesto, "Presupuesto_disponible"

# ---------- 6. RESOLVER EL MODELO ----------

# Le pedimos al solver (motor matemático) que encuentre la mejor solución
modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

# Estado de la solución: "Optimal" significa que sí se encontró la mejor respuesta
print("Estado:", pulp.LpStatus[modelo.status])

# Recorremos cada proyecto y mostramos si se debe ejecutar o no
for p in beneficios:
    print(f"{p}: {'Sí' if y[p].varValue == 1 else 'No'}")

# Mostramos el valor de la función objetivo (el beneficio máximo logrado)
print("Beneficio total (miles $):", pulp.value(modelo.objective))


Estado: Optimal
P1: Sí
P2: Sí
P3: No
P4: Sí
Beneficio total (miles $): 185.0


##  Ejercicio 2 — Asignación óptima de tareas a máquinas virtuales

**Enunciado:**
Se deben asignar 2 tareas (A y B) a 2 VMs (VM1 y VM2). Tiempos de ejecución (min):

|        | VM1 | VM2 |
|--------|-----|-----|
| Tarea A |  10 |  14 |
| Tarea B |  12 |   9 |

Cada tarea a una sola VM, cada VM procesa como máximo una tarea.
¿Qué asignación minimiza el tiempo total?


In [3]:
# ---------- 1. DATOS DEL PROBLEMA ----------

# Lista de tareas que hay que asignar
tareas = ["A", "B"]

# Lista de máquinas virtuales disponibles
vms = ["VM1", "VM2"]

# Diccionario con el tiempo (minutos) que tarda cada combinación tarea-VM
tiempo = {
    ("A", "VM1"): 10, ("A", "VM2"): 14,
    ("B", "VM1"): 12, ("B", "VM2"): 9,
}

# ---------- 2. CREAR EL MODELO ----------

# Creamos el problema; en este caso queremos MINIMIZAR el tiempo total
modelo = pulp.LpProblem("Asignacion_tareas_VMs", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# Una variable binaria por cada combinación (tarea, VM):
# x[t, v] = 1 si la tarea "t" se asigna a la VM "v", y 0 si no
x = {(t, v): pulp.LpVariable(f"Tarea_{t}_a_{v}", cat="Binary") for t in tareas for v in vms}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Minimizar la suma de los tiempos de las asignaciones que sí se realicen
modelo += pulp.lpSum(tiempo[t, v] * x[t, v] for t in tareas for v in vms), "Tiempo_total"

# ---------- 5. RESTRICCIONES ----------

# Cada tarea debe asignarse a exactamente UNA VM (ni más ni menos)
for t in tareas:
    modelo += pulp.lpSum(x[t, v] for v in vms) == 1, f"Tarea_{t}_asignada_una_vez"

# Cada VM puede recibir como máximo UNA tarea
for v in vms:
    modelo += pulp.lpSum(x[t, v] for t in tareas) <= 1, f"VM_{v}_una_tarea_maximo"

# ---------- 6. RESOLVER ----------

modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("Estado:", pulp.LpStatus[modelo.status])

# Recorremos todas las combinaciones (tarea, VM) y mostramos solo
# las que el modelo activó (valor = 1)
for (t, v), var in x.items():
    if var.varValue == 1:
        print(f"Tarea {t} -> {v}")

print("Tiempo total (min):", pulp.value(modelo.objective))


Estado: Optimal
Tarea A -> VM1
Tarea B -> VM2
Tiempo total (min): 19.0


##  Ejercicio 3 — Optimización de compra de licencias de software

**Enunciado:**
Se necesita cubrir a 130 empleados. Licencia Individual: 1 usuario, $20/mes.
Licencia Corporativa: 25 usuarios, $350/mes (solo unidades completas).
¿Cuántas licencias de cada tipo minimizan el costo mensual?


In [3]:
# ---------- 1. CREAR EL MODELO ----------
import pulp
# Queremos MINIMIZAR el costo mensual total
modelo = pulp.LpProblem("Optimizacion_licencias_software", pulp.LpMinimize)

# ---------- 2. VARIABLES DE DECISIÓN ----------

# Cantidad de Licencias Individuales a comprar (número entero, no puede ser negativo)
x1 = pulp.LpVariable("Licencias_Individuales", lowBound=0, cat="Integer")

# Cantidad de Licencias Corporativas a comprar (número entero, no puede ser negativo)
x2 = pulp.LpVariable("Licencias_Corporativas", lowBound=0, cat="Integer")

# ---------- 3. FUNCIÓN OBJETIVO ----------

# El costo total es $20 por cada licencia individual más $350 por cada licencia corporativa
modelo += 20 * x1 + 350 * x2, "Costo_mensual_total"

# ---------- 4. RESTRICCIÓN ----------

# La cantidad de usuarios cubiertos (1 por licencia individual, 25 por corporativa)
# debe ser al menos 130
modelo += x1 + 25 * x2 >= 130, "Cobertura_minima_usuarios"

# ---------- 5. RESOLVER ----------

modelo.solve()

# ---------- 6. MOSTRAR RESULTADOS ----------

print("Estado:", pulp.LpStatus[modelo.status])
print("Licencias Individuales:", x1.varValue)
print("Licencias Corporativas:", x2.varValue)
print("Costo mensual mínimo :", pulp.value(modelo.objective))


Estado: Optimal
Licencias Individuales: 5.0
Licencias Corporativas: 5.0
Costo mensual mínimo : 1850.0


##  Ejercicio 4 — Planificación óptima de turnos de soporte técnico

**Enunciado:**
La mesa de ayuda opera en 3 franjas al día con requerimientos mínimos de técnicos:

| Franja | Mínimo |
|---|---|
| Mañana | 4 |
| Tarde | 6 |
| Noche | 3 |

Turno Diurno (cubre Mañana y Tarde): $60/día. Turno Nocturno (cubre Tarde y Noche): $70/día.
¿Cuántos técnicos de cada turno minimizan el costo?


In [5]:
# ---------- 1. CREAR EL MODELO ----------

# Queremos MINIMIZAR el costo diario total
modelo = pulp.LpProblem("Planificacion_turnos_soporte", pulp.LpMinimize)

# ---------- 2. VARIABLES DE DECISIÓN ----------

# Número de técnicos contratados en Turno Diurno (entero, no negativo)
x1 = pulp.LpVariable("Turno_Diurno", lowBound=0, cat="Integer")

# Número de técnicos contratados en Turno Nocturno (entero, no negativo)
x2 = pulp.LpVariable("Turno_Nocturno", lowBound=0, cat="Integer")

# ---------- 3. FUNCIÓN OBJETIVO ----------

# El costo diario es $60 por cada técnico en Turno Diurno
# más $70 por cada técnico en Turno Nocturno
modelo += 60 * x1 + 70 * x2, "Costo_diario_total"

# ---------- 4. RESTRICCIONES ----------

# En la Mañana solo trabaja el Turno Diurno, así que x1 debe cubrir el mínimo de 4
modelo += x1 >= 4, "Cobertura_Mañana"

# En la Tarde trabajan ambos turnos, la suma debe cubrir el mínimo de 6
modelo += x1 + x2 >= 6, "Cobertura_Tarde"

# En la Noche solo trabaja el Turno Nocturno, así que x2 debe cubrir el mínimo de 3
modelo += x2 >= 3, "Cobertura_Noche"

# ---------- 5. RESOLVER ----------

modelo.solve()

# ---------- 6. MOSTRAR RESULTADOS ----------

print("Estado:", pulp.LpStatus[modelo.status])
print("Turno Diurno:", x1.varValue)
print("Turno Nocturno:", x2.varValue)
print("Costo mínimo diario ($):", pulp.value(modelo.objective))


Estado: Optimal
Turno Diurno: 4.0
Turno Nocturno: 3.0
Costo mínimo diario ($): 450.0
